In [1]:
#XGBoost
sc.install_pypi_package("xgboost")
sc.install_pypi_package("scikit-learn")
sc.install_pypi_package("pandas")
sc.install_pypi_package("numpy")

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
8,application_1781028837408_0009,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…



  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.8.1
    Not uninstalling python-dateutil at /usr/lib/python3.9/site-packages, outside environment /mnt/yarn/usercache/livy/appcache/application_1781028837408_0009/container_1781028837408_0009_01_000001/tmp/spark-c43f1abd-5f7b-42c8-a94b-80ce5a68ad44
    Can't uninstall 'python-dateutil'. No files were found to uninstall.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas 2.3.3 requires tzdata>=2022.7, which is not installed.
matplotlib 3.9.4 requires contourpy>=1.0.1, which is not installed.
matplotlib 3.9.4 requires cycler>=0.10, which is not installed.
matplotlib 3.9.4 requires fonttools>=4.22.0, which is not installed.
matplotlib 3.9.4 requires importlib-resources>=3.2.0; python_version < "3.10", which is not installed.
matplotlib 3.9.4 requires kiwisolver>=1.3.

In [2]:
df_saved = spark.read.parquet(
    "s3://csc555-data-dd9d74e8/processed/final_features/"
)

print("Rows:", df_saved.count())
print("Columns:", len(df_saved.columns))
df_saved.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Rows: 992931
Columns: 24
root
 |-- msno: string (nullable = true)
 |-- is_churn: integer (nullable = true)
 |-- city: integer (nullable = true)
 |-- bd: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- registered_via: integer (nullable = true)
 |-- registration_init_time: integer (nullable = true)
 |-- transaction_count: long (nullable = true)
 |-- avg_payment_plan_days: double (nullable = true)
 |-- avg_plan_list_price: double (nullable = true)
 |-- avg_actual_amount_paid: double (nullable = true)
 |-- auto_renew_count: long (nullable = true)
 |-- cancel_count: long (nullable = true)
 |-- last_transaction_date: integer (nullable = true)
 |-- last_membership_expire_date: integer (nullable = true)
 |-- log_days: long (nullable = true)
 |-- avg_num_25: double (nullable = true)
 |-- avg_num_50: double (nullable = true)
 |-- avg_num_75: double (nullable = true)
 |-- avg_num_985: double (nullable = true)
 |-- avg_num_100: double (nullable = true)
 |-- avg_num_unq: double

In [3]:
feature_cols = [
    "city",
    "bd",
    "registered_via",
    "transaction_count",
    "avg_payment_plan_days",
    "avg_plan_list_price",
    "avg_actual_amount_paid",
    "auto_renew_count",
    "cancel_count",
    "log_days",
    "avg_num_25",
    "avg_num_50",
    "avg_num_75",
    "avg_num_985",
    "avg_num_100",
    "avg_num_unq",
    "avg_total_secs",
    "sum_total_secs"
]

pdf = df_saved.select(
    feature_cols + ["is_churn"]
).toPandas()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
print(df_saved.count())
print(pdf.shape)
print(pdf.head())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

992931
(992931, 19)
   city  bd  registered_via  ...  avg_total_secs  sum_total_secs  is_churn
0    14  31               7  ...     2340.949887    1.032359e+06         0
1     1   0               7  ...     9824.775080    2.957257e+06         0
2     1   0               7  ...     1961.379000    1.961379e+03         0
3    22  32               9  ...     4104.195538    7.633804e+05         0
4    13  29               9  ...    39060.542332    3.070159e+07         0

[5 rows x 19 columns]

In [7]:
from sklearn.model_selection import train_test_split

X = pdf[feature_cols]
y = pdf["is_churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

X_train: (794344, 18)
X_test: (198587, 18)
y_train: (794344,)
y_test: (198587,)

In [8]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    random_state=42
)

xgb_model.fit(X_train, y_train)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost Results")
print("AUC =", roc_auc_score(y_test, y_prob))
print("Accuracy =", accuracy_score(y_test, y_pred))
print("Precision =", precision_score(y_test, y_pred))
print("Recall =", recall_score(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

XGBoost Results
AUC = 0.9550333947240552
Accuracy = 0.9507218498693268
Precision = 0.7243827160493828
Recall = 0.36978099889711674
Confusion Matrix:
[[184107   1786]
 [  8000   4694]]

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix

y_prob = xgb_model.predict_proba(X_test)[:, 1]

y_pred_03 = (y_prob >= 0.3).astype(int)

print("XGBoost Results with threshold = 0.3 because the data is imbalanced")
print("AUC =", roc_auc_score(y_test, y_prob))
print("Accuracy =", accuracy_score(y_test, y_pred_03))
print("Precision =", precision_score(y_test, y_pred_03))
print("Recall =", recall_score(y_test, y_pred_03))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_03))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

XGBoost Results with threshold = 0.3
AUC = 0.9550333947240552
Accuracy = 0.9433447305211318
Precision = 0.5526603897525728
Recall = 0.596502284543879
Confusion Matrix:
[[179764   6129]
 [  5122   7572]]

In [11]:
# Threshhold tuning
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_prob = xgb_model.predict_proba(X_test)[:, 1]

thresholds = [0.2, 0.25, 0.3, 0.35, 0.4, 0.5]

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)

    print("Threshold:", t)
    print("Accuracy:", accuracy_score(y_test, y_pred_t))
    print("Precision:", precision_score(y_test, y_pred_t))
    print("Recall:", recall_score(y_test, y_pred_t))
    print("F1:", f1_score(y_test, y_pred_t))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred_t))
    print("------------------------")
# best threshold is 0.3

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Threshold: 0.2
Accuracy: 0.9193048890410752
Precision: 0.42904613811613346
Recall: 0.7933669450133921
F1: 0.5569165261149667
Confusion Matrix:
[[172491  13402]
 [  2623  10071]]
------------------------
Threshold: 0.25
Accuracy: 0.9345375074904199
Precision: 0.4913919207831664
Recall: 0.6880415944540728
F1: 0.5733228305106998
Confusion Matrix:
[[176853   9040]
 [  3960   8734]]
------------------------
Threshold: 0.3
Accuracy: 0.9433447305211318
Precision: 0.5526603897525728
Recall: 0.596502284543879
F1: 0.5737450274673234
Confusion Matrix:
[[179764   6129]
 [  5122   7572]]
------------------------
Threshold: 0.35
Accuracy: 0.9478415001989052
Precision: 0.6081681792924616
Recall: 0.5173310225303293
F1: 0.5590839434701175
Confusion Matrix:
[[181662   4231]
 [  6127   6567]]
------------------------
Threshold: 0.4
Accuracy: 0.9495737384622357
Precision: 0.648921982662814
Recall: 0.4599810934299669
F1: 0.5383551539738153
Confusion Matrix:
[[182734   3159]
 [  6855   5839]]
--------------

In [12]:
# class imbalance weight
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Negative count:", negative_count)
print("Positive count:", positive_count)
print("scale_pos_weight:", scale_pos_weight)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Negative count: 743567
Positive count: 50777
scale_pos_weight: 14.643775725229927

In [13]:
# Weighted XGBoost
from xgboost import XGBClassifier

xgb_weighted = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=14.64,
    eval_metric="auc",
    random_state=42
)

xgb_weighted.fit(X_train, y_train)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

y_prob_w = xgb_weighted.predict_proba(X_test)[:, 1]
y_pred_w = (y_prob_w >= 0.5).astype(int)

print("Weighted XGBoost Results")
print("AUC:", roc_auc_score(y_test, y_prob_w))
print("Accuracy:", accuracy_score(y_test, y_pred_w))
print("Precision:", precision_score(y_test, y_pred_w))
print("Recall:", recall_score(y_test, y_pred_w))
print("F1:", f1_score(y_test, y_pred_w))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_w))

# XGBoost performed slightly better, with an AUC of 0.955. The default threshold of 0.5 gave
# high precision but low recall, meaning the model only predicted churn when it was very confident and 
# missed many churn users. After lowering the threshold to 0.3, recall improved and the F1 score became
# more balanced. The weighted XGBoost model caught more churn uesrs, but it also created too many false
# positives, so it was not necessarily better overall. It is better to flag more customers as 
# potential churners than to miss customers who are actually likely to churn.
# conclusion
# Best AUC:
# XGBoost = 0.9550

# Best F1:
# XGBoost (threshold=0.3) = 0.574

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Weighted XGBoost Results
AUC: 0.9549186434649658
Accuracy: 0.8624532320846783
Precision: 0.31120064047932644
Recall: 0.9492673704112179
F1: 0.46873480501799086
Confusion Matrix:
[[159222  26671]
 [   644  12050]]